In [49]:
def clean_northern_farmer_names():
    
    # Import relevant dependencies
    import pandas as pd
    
    """
    Clean Northern farmers' data for ingestion.
    Dumps csv.
    Returns Pandas Dataframe.    
    """
    # Load Excel file
    df = pd.read_excel('data/original_dataset/farmer_data.xlsx')
    # Rename columns
    df.columns = ['name',
                  'telephone_no',
                  'crops_planted',
                  'yet_to_plant',
                  'harvest_date',
                  'location',
                  'district',
                  'region']
    # Select object data types    
    object_columns = df.select_dtypes(include='object').columns.to_list()

    # Clean and standardize
    for column in object_columns:
        missing = 'Nothing'
        df[column] = df[column].fillna(missing)
        df[column] = df[column].apply(lambda x: x.title())

    # Save the file
    df.to_csv('data/original_dataset/farmer_data.csv')
    return df

In [56]:
def ingest_northern_farmer_names(df: pd.DataFrame = clean_northern_farmer_names()) -> pd.DataFrame:

    # Import relevant dependencies
    import os
    from dotenv import load_dotenv
    from sqlalchemy import create_engine

    """
    Calls the clean_northern_farmer_names function and ingests the data into a MySQL database.
    
    Parameters:
    df -> DataFrame: Pandas DataFrame

    Returns None.
    """
    
    # Database configuration
    username = os.getenv('DB_USERNAME')
    host = os.getenv('DB_HOST')
    port = os.getenv('DB_PORT')
    database = os.getenv('DB_DATABASE')
    password = os.getenv('DB_PASSWORD_F')

    # Create database connection
    if port is None or port == 'None':
        connection_string = f'mysql+pymysql://{username}:{password}@{host}/{database}'
    else:
        connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
    
    engine = create_engine(connection_string)

    # Save to SQL
    df.to_sql(name='new_farmers',        # Name of the SQL table
              con=engine,                # Database connection engine
              if_exists='replace',       # Options: 'fail', 'replace', 'append'
              index=False)               # Do not write the DataFrame index as a column
    print('Ingestion done. ✅')

In [57]:
ingest_northern_farmer_names()

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'None' ([Errno 11001] getaddrinfo failed)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)